In [1]:
# %pip install pymupdf python-docx
# %pip install spacy
# %pip install spacy-transformers
# For the high-accuracy Transformer model (F1 ~90%)
# %pip install https://huggingface.co/opennyaiorg/en_legal_ner_trf/resolve/main/en_legal_ner_trf-any-py3-none-any.whl

In [2]:
import fitz  # PyMuPDF
import re
import os
from docx import Document
import tkinter as tk
from tkinter import filedialog
import sys

from pathlib import Path

In [3]:
def legal_cleaning(text):
    """
    Enhanced cleaning for Indian Law Reports: 
    Removes margin ladders (A-H), page headers, and numbers 
    while preserving structure for Legal-BERT.
    """
    # 1. Remove "Alphabet Ladders" (Standalone A-H on their own lines)
    # These appear in the margins of court reporters [cite: 1, 6, 209]
    text = re.sub(r'^\s*[A-H]\s*$', '', text, flags=re.MULTILINE)
    
    # 2. Remove Page Headers/Footers [cite: 109, 110]
    # Targeted patterns found in your specific document
    headers = [
        r'\[2020\]\s*1\s*S\.C\.R\.', 
        r'SUPREME\s*COURT\s*REPORTS',
        r'UNION\s*OF\s*INDIA\s*v\.\s*RELIANCE\s*COMMUNICATION.*',
        r'S\.\s*RAVINDRA\s*BHAT,\s*J\.'
    ]
    for pattern in headers:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 3. Remove Standalone Page Numbers [cite: 108, 138, 189]
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 4. Standard Cleaning Logic (Preserved from your original code)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r' +', ' ', text)
    text = "".join(char for char in text if char.isprintable() or char in ['\n', '\t'])
    
    # Final trim to ensure no trailing blank lines from the removed ladders
    text = "\n".join([line.strip() for line in text.split('\n') if line.strip()])
    
    return text.strip()

In [4]:
def process_document():
    # 2. Interactive File Selection
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True) 
    
    # FIX 1: Define the allowed file types in a list of tuples
    file_path = filedialog.askopenfilename(
        title="Layer 1: Select Legal Document (PDF or Word)",
        filetypes=[
            ("PDF files", "*.pdf"),
            ("Word documents", "*.docx"),
            ("All files", "*.*")
        ]
    )

    if not file_path:
        print("Selection cancelled.")
        return None

    # FIX 2: Corrected the syntax for getting the extension
    # It was: os.path.splitext(file_path).[1]lower()
    ext = os.path.splitext(file_path)[1].lower() 
    
    print(f"Extracting: {os.path.basename(file_path)}...")

    # Get the file name and remove .pdf from the og filename
    fileName = str(os.path.basename(file_path)).replace(".pdf", "")

    # 3. Text Extraction
    raw_text = ""
    try:
        if ext == ".pdf":
            with fitz.open(file_path) as doc:
                raw_text = "\n".join([page.get_text("text") for page in doc])
        elif ext == ".docx":
            doc = Document(file_path)
            raw_text = "\n".join([para.text for para in doc.paragraphs])
        
        # 4. Pre-processing & Cleaning
        cleaned_text = legal_cleaning(raw_text)

        # 5. Output for later stages
        folder_name = "Text_Extracts"
        os.makedirs(folder_name, exist_ok=True)   # Creates folder if it doesn't exis   
        output_file = os.path.join(folder_name, f"{fileName}.txt")

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"Success! Cleaned text saved to: {output_file}")
        print(f"Character count: {len(cleaned_text)}")
        return cleaned_text

    except Exception as e:
        print(f"Extraction error: {str(e)}")
        return None

In [5]:
cleaned_data = process_document() 


Extracting: 2020_1_8_18_EN.pdf...
Success! Cleaned text saved to: Text_Extracts\2020_1_8_18_EN.txt
Character count: 25307


The Segmentation method for extracting Evidence, Facts and IPCs from cleaned Text file

In [6]:
from caller import extract_legal_details

In [7]:
# Get the extracted facts, evidence and basic IPCs from the text file
result = extract_legal_details(cleaned_data)
print(result)

{'FACTS': "The appellant, acting as a Power of Attorney holder for the property owners, leased a commercial premises to Respondent No. 1 (Bala Venkatram) in May 2007 for running a business named 'Best Mark Super Market'. Following a default in rent and a change in the shop's name to 'Amutham Super Market', the appellant discovered that Respondent No. 1 had effectively handed over the premises to Respondent No. 2 (Shahu Hameed). The appellant filed an eviction petition on grounds of sub-letting and arrears of rent. While the respondents claimed an oral partnership existed between them, the Rent Control Appellate Authority granted eviction, which was later set aside by the High Court. The Supreme Court was tasked with determining if a genuine partnership existed or if it was an unauthorized sub-letting.", 'EVIDENCE': ['Rental Agreement dated 23.05.2007 between the appellant and respondent no. 1.', 'Power of Attorney dated 01.11.2006 executed in favor of the appellant.', 'Sales Tax Certif

In [8]:
print(f"The processed facts using gemini 3 flash are:")
case_facts = result.get('FACTS')
print(case_facts)

The processed facts using gemini 3 flash are:
The appellant, acting as a Power of Attorney holder for the property owners, leased a commercial premises to Respondent No. 1 (Bala Venkatram) in May 2007 for running a business named 'Best Mark Super Market'. Following a default in rent and a change in the shop's name to 'Amutham Super Market', the appellant discovered that Respondent No. 1 had effectively handed over the premises to Respondent No. 2 (Shahu Hameed). The appellant filed an eviction petition on grounds of sub-letting and arrears of rent. While the respondents claimed an oral partnership existed between them, the Rent Control Appellate Authority granted eviction, which was later set aside by the High Court. The Supreme Court was tasked with determining if a genuine partnership existed or if it was an unauthorized sub-letting.


In [9]:
print(f"The processed evidence using gemini 3 flash are:")
case_evidence = result.get('EVIDENCE')
print(case_evidence)

The processed evidence using gemini 3 flash are:
['Rental Agreement dated 23.05.2007 between the appellant and respondent no. 1.', 'Power of Attorney dated 01.11.2006 executed in favor of the appellant.', 'Sales Tax Certificate and Registration in the name of respondent no. 2.', 'Shop License issued in the name of respondent no. 2.', 'Bank accounts of the firm showing respondent no. 2 as the sole owner.', 'Cross-examination of Respondent No. 1 where he admitted handing over the premises to respondent no. 2.', 'Legal notice issued by the landlady to the original tenant.', 'Absence of any written partnership deed or document proving a joint business venture.']


In [10]:
print(f"The processed IPCs using gemini 3 flash are:")
case_ipc = result.get('IPC/STATUTES')
print(case_ipc)

The processed IPCs using gemini 3 flash are:
['Tamil Nadu Buildings (Lease and Rent Control) Act, 1960', 'Section 10(2)(i) (Eviction for arrears of rent)', 'Section 10(2)(ii)(a)(b) (Eviction for sub-letting/transfer of rights)', 'Section 10(2)(iii) (Eviction for wastage or material alteration)', 'Section 2(6) (Definition of Landlord)']


Call the Evidence Scrutinization Llama 3 model from HF

In [11]:
# Use getcwd() instead of __file__
current_dir = os.getcwd()
target_path = os.path.abspath(os.path.join(current_dir, "..", "2_Evidence_Scrutinization"))

if target_path not in sys.path:
    sys.path.append(target_path)

from scrutiny import scrutinize_legal_data

In [12]:
from scrutiny import scrutinize_legal_data

In [13]:
scr_result = scrutinize_legal_data(case_facts, case_evidence, case_ipc)

Server returned error 404: Not Found


In [14]:
print(scr_result)

Error: Server returned status 404
